# 05 — Model Comparison

## Objective
Compare multiple machine-learning algorithms against our Logistic Regression baseline. The objective is to improve fraud detection and risk ranking, especially under the investigation-capacity constraint.

**Primary business metric:** Precision@K, where K is scaled to the test-set share of the assumed monthly capacity of 200 claims. **Secondary:** Recall@K. We also compare standard classification metrics at the 0.50 threshold.

This dataset is historical and used as an educational portfolio case. Capacity metrics are a simulation, not a real-world operational result.

## 1. Imports and locate project

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

warnings.filterwarnings("ignore", category=FutureWarning)

search_start = Path.cwd().resolve()
project_root = None
for candidate in [search_start, *search_start.parents]:
    if (candidate / "data" / "processed" / "X_train.csv").exists():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError("Could not find data/processed/X_train.csv from current directory")

DATA_DIR = project_root / "data" / "processed"
MODELS_DIR = project_root / "models"
REPORTS_DIR = project_root / "reports"
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)
print("Project root:", project_root)

Project root: C:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection


## 2. Load train/test data
Use the existing stratified split. Do not re-split or fit preprocessing on the test set.

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").iloc[:, 0].astype(int)
y_test = pd.read_csv(DATA_DIR / "y_test.csv").iloc[:, 0].astype(int)

assert X_train.columns.tolist() == X_test.columns.tolist()
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Train fraud prevalence: {y_train.mean():.3%}")
print(f"Test fraud prevalence: {y_test.mean():.3%}")

Train: (12336, 42)  Test: (3084, 42)
Train fraud prevalence: 5.982%
Test fraud prevalence: 5.999%


## 3. Feature type identification and preprocessing
Each candidate uses the same preprocessing logic. Numeric features are median-imputed and scaled. Categorical features are imputed with the most frequent value and one-hot encoded. All preprocessing is fit inside each model pipeline using only training data.

In [3]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features)
    ])

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Numeric features: 17
Categorical features: 25


## 4. Define candidate models
We compare three candidates:
1. Logistic Regression — standard baseline (no class weights).
2. Logistic Regression — balanced class weights, giving the minority class more influence.
3. Random Forest — class-weighted tree ensemble, able to capture nonlinearities and interactions.


These are initial comparison settings, not hyperparameter-tuned models. We keep model tuning and threshold optimization for later tasks.

In [4]:
models = {
    "Logistic Regression (baseline)": LogisticRegression(max_iter=2000, random_state=42),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Random Forest (balanced)": RandomForestClassifier(
        n_estimators=100, class_weight="balanced_subsample",
        min_samples_leaf=2, n_jobs=1, random_state=42
    )
}
models

{'Logistic Regression (baseline)': LogisticRegression(max_iter=2000, random_state=42),
 'Logistic Regression (balanced)': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
 'Random Forest (balanced)': RandomForestClassifier(class_weight='balanced_subsample', min_samples_leaf=2,
                        n_jobs=1, random_state=42)}

## 5. Train and evaluate candidates
We calculate threshold-based metrics at 0.50 and ranking metrics at capacity-scaled K. This keeps the business evaluation consistent across models. We also capture each model’s confusion matrix.

In [5]:
results = []
trained_models = {}
predictions = {}
confusion_matrices = {}

capacity_monthly = 200
test_fraction = len(y_test) / (len(y_train) + len(y_test))
k_test = max(1, int(round(capacity_monthly * test_fraction)))

for model_name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessing", make_preprocessor()),
        ("model", estimator)
    ])
    print(f"Training: {model_name}")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    ranking = pd.DataFrame({
        "actual": y_test.reset_index(drop=True),
        "probability": prob
    }).sort_values("probability", ascending=False)
    top_k = ranking.head(k_test)
    precision_k = float(top_k["actual"].mean())
    recall_k = float(top_k["actual"].sum() / y_test.sum()) if y_test.sum() else 0.0

    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_0_50": precision_score(y_test, pred, zero_division=0),
        "Recall_0_50": recall_score(y_test, pred, zero_division=0),
        "F1_0_50": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "PR_AUC_Average_Precision": average_precision_score(y_test, prob),
        "Test_Top_K": k_test,
        "Fraud_in_Top_K": int(top_k["actual"].sum()),
        "Precision_at_K": precision_k,
        "Recall_at_K": recall_k
    }
    results.append(row)
    trained_models[model_name] = pipe
    predictions[model_name] = {"pred": pred, "prob": prob}
    confusion_matrices[model_name] = confusion_matrix(y_test, pred, labels=[0, 1])

comparison = pd.DataFrame(results)
comparison


Training: Logistic Regression (baseline)
Training: Logistic Regression (balanced)
Training: Random Forest (balanced)


,Model,Accuracy,Precision_0_50,Recall_0_50,F1_0_50,ROC_AUC,PR_AUC_Average_Precision,Test_Top_K,Fraud_in_Top_K,Precision_at_K,Recall_at_K
0,Logistic Regression (baseline),0.940337,0.666667,0.010811,0.021277,0.816289,0.177789,40,8,0.20,0.043243
1,Logistic Regression (balanced),0.646887,0.131922,0.875676,0.229299,0.807581,0.152762,40,6,0.15,0.032432
2,Random Forest (balanced),0.941310,0.700000,0.037838,0.071795,0.828253,0.231731,40,16,0.40,0.086486


## 6. Compare results
Sort by Precision@K first because the scenario prioritizes precision within a fixed investigation capacity. Recall@K is the next business metric. ROC-AUC and PR-AUC provide broader discrimination/ranking context. Threshold-based metrics are included but should not be the sole selection criteria.

In [6]:
comparison_sorted = comparison.sort_values(
    ["Precision_at_K", "Recall_at_K", "PR_AUC_Average_Precision"],
    ascending=False
).reset_index(drop=True)
comparison_sorted.round(4)

,Model,Accuracy,Precision_0_50,Recall_0_50,F1_0_50,ROC_AUC,PR_AUC_Average_Precision,Test_Top_K,Fraud_in_Top_K,Precision_at_K,Recall_at_K
0,Random Forest (balanced),0.9413,0.7000,0.0378,0.0718,0.8283,0.2317,40,16,0.40,0.0865
1,Logistic Regression (baseline),0.9403,0.6667,0.0108,0.0213,0.8163,0.1778,40,8,0.20,0.0432
2,Logistic Regression (balanced),0.6469,0.1319,0.8757,0.2293,0.8076,0.1528,40,6,0.15,0.0324


## 7. Compare against random selection
Random selection is expected to yield fraud precision approximately equal to the fraud prevalence in the test set. This baseline helps contextualize whether ranking is adding value.

In [7]:
random_precision = float(y_test.mean())
capacity_summary = comparison_sorted[[
    "Model", "Test_Top_K", "Fraud_in_Top_K", "Precision_at_K", "Recall_at_K"
]].copy()
capacity_summary["Random_Precision_Baseline"] = random_precision
capacity_summary["Precision_Lift_vs_Random"] = capacity_summary["Precision_at_K"] / random_precision
capacity_summary.round(4)

,Model,Test_Top_K,Fraud_in_Top_K,Precision_at_K,Recall_at_K,Random_Precision_Baseline,Precision_Lift_vs_Random
0,Random Forest (balanced),40,16,0.40,0.0865,0.06,6.6681
1,Logistic Regression (baseline),40,8,0.20,0.0432,0.06,3.3341
2,Logistic Regression (balanced),40,6,0.15,0.0324,0.06,2.5005


## 8. Inspect confusion matrices
The confusion matrix helps understand the fixed-threshold (0.50) trade-off for each model. The investigation team, however, will ultimately use ranked claims and capacity-based selection.

In [8]:
for name, cm in confusion_matrices.items():
    print("\n", name)
    print(pd.DataFrame(cm, index=["Actual Legitimate", "Actual Fraud"],
                       columns=["Predicted Legitimate", "Predicted Fraud"]))


 Logistic Regression (baseline)
                   Predicted Legitimate  Predicted Fraud
Actual Legitimate                  2898                1
Actual Fraud                        183                2

 Logistic Regression (balanced)
                   Predicted Legitimate  Predicted Fraud
Actual Legitimate                  1833             1066
Actual Fraud                         23              162

 Random Forest (balanced)
                   Predicted Legitimate  Predicted Fraud
Actual Legitimate                  2896                3
Actual Fraud                        178                7


## 9. Identify the current leading candidate
This selection is provisional and based on the held-out test results. Model selection using the test set can create optimistic bias if repeated extensively; subsequent tuning should ideally use cross-validation on training data, with the test set reserved for final confirmation.

In [9]:
best_model_name = comparison_sorted.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Provisional leader based on Precision@K, then Recall@K, then PR-AUC:", best_model_name)
print(comparison_sorted.iloc[0].to_dict())

Provisional leader based on Precision@K, then Recall@K, then PR-AUC: Random Forest (balanced)
{'Model': 'Random Forest (balanced)', 'Accuracy': 0.9413099870298314, 'Precision_0_50': 0.7, 'Recall_0_50': 0.03783783783783784, 'F1_0_50': 0.07179487179487179, 'ROC_AUC': 0.8282529856520888, 'PR_AUC_Average_Precision': 0.2317311743813058, 'Test_Top_K': 40, 'Fraud_in_Top_K': 16, 'Precision_at_K': 0.4, 'Recall_at_K': 0.08648648648648649}


## 10. Save comparison artifacts
Save the comparison table, capacity-focused summary, individual confusion matrices, and all trained pipelines. These artifacts will support subsequent threshold optimization and explainability work.

In [10]:
comparison_path = REPORTS_DIR / "model_comparison_metrics.csv"
capacity_path = REPORTS_DIR / "model_comparison_capacity_metrics.csv"
comparison_sorted.to_csv(comparison_path, index=False)
capacity_summary.to_csv(capacity_path, index=False)

for name, cm in confusion_matrices.items():
    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    cm_df = pd.DataFrame(cm, index=["Actual_Legitimate", "Actual_Fraud"],
                         columns=["Predicted_Legitimate", "Predicted_Fraud"])
    cm_df.to_csv(REPORTS_DIR / f"confusion_matrix_{safe_name}.csv")

for name, pipe in trained_models.items():
    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    joblib.dump(pipe, MODELS_DIR / f"{safe_name}.joblib")

print("Saved comparison:", comparison_path)
print("Saved capacity comparison:", capacity_path)
print("Saved candidate pipelines in:", MODELS_DIR)

Saved comparison: C:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection\reports\model_comparison_metrics.csv
Saved capacity comparison: C:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection\reports\model_comparison_capacity_metrics.csv
Saved candidate pipelines in: C:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection\models


### A. Balanced Logistic Regression

Class weighting dramatically improves fraud recall at the default threshold:
- Recall increases from 1.08% → 87.57%
- But precision falls from 66.67% → 13.18%
- Accuracy falls from 94.03% → 64.66%
- Precision@40 actually decreases from 20% → 15%

This is an important lesson: improving recall at a fixed threshold doesn't necessarily improve the model's ranking under a fixed investigation capacity.

The balanced model flags many more claims as fraudulent, but its ranking isn't the best for selecting a limited number of claims.

### B. Random Forest

Random Forest gives us the strongest business-focused result in this experiment:
- Best ROC-AUC: 0.8260
- Best PR-AUC: 0.2268
- Best Precision@40: 37.50%
- Finds 15 frauds in the Top 40, compared with 8 for baseline Logistic Regression.

This suggests that the tree-based model may be capturing nonlinear relationships and interactions useful for ranking claims.

### C. Why the balanced Logistic Regression isn't our winner

Although it achieves extremely high recall at threshold 0.50, our investigation team cannot investigate every claim it flags.